In [1]:
# Import functions from preprocessing.py
import sys
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.optim as optim

from VIGP_Unlinked_diffpiX_diffpiS import VIGP_Unlinked
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from GPModel import GPModel
from GPArealModel import GPArealModel


# Add the path to the src directory
sys.path.append(os.path.abspath(os.path.join('..', 'data')))

result = {}
B = 49
n_i = 4
seed = 1
input_dim = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

niter_GP=10
niter_GPAreal=10
niter_VI= 50

# Load data from the specified path
data_path = os.path.abspath(os.path.join('..', '..', '..', 'data', 'data_diff_piX_piS', f'B_{B}_n_{n_i}', f'data_seed_{seed}.pt'))
data = torch.load(data_path)

# Extract variables from the data dictionary
y = data['y']
region_assignments = data['region_assignments']
x = data['x']
w = data['w']
e = data['e']
s = data['s']
x_jumbled_within_regions = data['x_jumbled_within_regions']
s_jumbled_within_regions = data['s_jumbled_within_regions']
perm_matrices_x = data['perm_matrices_x']
perm_matrices_s = data['perm_matrices_s']
sigmasq_true = data['sigmasq_true']
phi_true = data['phi_true']
beta_true = data['beta_true']
nu_true = data['nu_true']
tausq_true = data['tausq_true']

# Train the GPmodel (oracle)
# Oracle GP model if the locations and links are known
# Initialize and optimize the model
model = GPModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(niter_GP)):
    optimizer.zero_grad()
    loss = model(s, x, y)
    loss.backward()
    optimizer.step()
    
    # Constrain sigmasq, length_scale, and tausq to be positive
    with torch.no_grad():
        model.sigmasq.clamp_(min=1e-6)
        model.phi.clamp_(min=1e-6)
        model.tausq.clamp_(min=1e-6)
        
# Save the model parameters
model_params = {
    'nu': model.nu.item(),
    'phi': model.phi.item(),
    'sigmasq': model.sigmasq.item(),
    'tausq': model.tausq.item(),
    'beta': model.beta.detach().cpu().numpy()
}

# Save the model parameters to a file
result['GPmodel'] = model_params


# Train the model GPareal
# Compute region-wise averages directly
unique_regions = torch.unique(region_assignments)
B = len(unique_regions)

ybar = torch.zeros(B, device=y.device)
xbar = torch.zeros(B, input_dim, device=x.device)

for i, region in enumerate(unique_regions):
    # Get indices for the current region
    indices = torch.where(region_assignments == region)[0]
    
    # Compute region-wise averages for y and x
    ybar[i] = torch.mean(y[indices])
    xbar[i] = torch.mean(x_jumbled_within_regions[indices], dim=0)


# Initialize and optimize the model
model = GPArealModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(niter_GPAreal)):
    optimizer.zero_grad()
    loss = model(s_jumbled_within_regions, region_assignments, xbar, ybar)
    loss.backward()
    optimizer.step()


# Save the model parameters
model_params = {
    'nu': model.nu.item(),
    'phi': model.phi.item(),
    'sigmasq': model.sigmasq.item(),
    'tausq': model.tausq.item(),
    'beta': model.beta.detach().cpu().numpy()
}
# Save the model parameters to a file
result['GPArealModel'] = model_params


# Train the model VIGP_unlinked
n_blocks = B
n_locations = n_i

# Random toy data for X and Y
X = torch.tensor(x_jumbled_within_regions, dtype=torch.float32).reshape(n_blocks, n_locations)
Y = torch.tensor(y, dtype=torch.float32).reshape(n_blocks, n_locations)

# Generate (n_blocks * n_locations) 2D coordinates
total_points = n_blocks * n_locations
locations = torch.tensor(s_jumbled_within_regions,dtype=torch.float32)

# Compute distance matrix from locations
Dist = torch.cdist(locations, locations, p=2)  # Pairwise distances
Dist = (Dist + Dist.T) / 2  # Make it symmetric because numerical errors can cause asymmetry

# Set optional args
n_steps = 10
n_phi_samples = 100
n_piX_sample = 50
tau_X = 0.3
tau_S = 0.3
n_piS_sample = 50

for tau in [0.3]:
    tau_X = tau
    tau_S = tau

    results_VI = VIGP_Unlinked(
        n_iter=niter_VI,
        n_blocks=n_blocks,
        n_locations=n_locations,
        phi_prior_ub=5,
        X=X,
        Y=Y,
        Dist=Dist,
        n_steps=n_steps,
        n_phi_samples=n_phi_samples,
        n_piX_sample=n_piX_sample,
        tau_X=tau_X, tau_S=tau_S,
        n_piS_sample=n_piS_sample,
        seed=521, 
        fix_piX= False, 
        fix_piS= False,
        fix_mu_lambda_beta=False,
        fix_sigmasq_lambda_beta=False,
        fix_lambda_b1=False,
        lambda_b1_fixed=((B*n_i)*0.5 + 0.1) * 5,
        fix_lambda_b2=False,
        M_X_star_fixed=[_.T for _ in perm_matrices_x],
        M_S_star_fixed=[_.T for _ in perm_matrices_s],
        V_X_star_fixed=[torch.eye(n_locations, n_locations, device=device) for _ in range(n_blocks)],
        V_S_star_fixed=[torch.eye(n_locations, n_locations, device=device) for _ in range(n_blocks)], 
        phi_init = 0.5,
        mean_Rphi_inv_fixed= torch.linalg.inv(torch.exp(-4 * Dist)),
        fix_mean_Rphi_inv=False, 
        pi_X_true = perm_matrices_x,
        pi_S_true = perm_matrices_s,
        VX_ub = 0.5,
        VS_ub=0.5,
        lr_piX = 0.1,
        lr_piS = 0.1
    )
   
    # Save the model parameters to the result dictionary
    result[f'VIGP_unlinked_tau_{tau}'] = results_VI

result0 = result.copy()
# Save the result dictionary to a file
# result_path = os.path.join('..', 'data', 'results' , f'B_{B}_n_{n_i}', f'results_seed_{seed}.pt')
# os.makedirs(os.path.dirname(result_path), exist_ok=True)
# torch.save(result, result_path)

/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_53889/3600119368.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(data_path)
  0%|          

Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 1.1921e-07
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.1921e-07
Block 0, Step 0, Loss: 39.9267
Block 0, Step 1, Loss: 39.2242
Block 0, Step 2, Loss: 38.5883
Block 0, Step 3, Loss: 37.9782
Block 0, Step 4, Loss: 37.2644
Block 0, Step 5, Loss: 36.5946
Block 0, Step 6, Loss: 35.9887
Block 0, Step 7, Loss: 35.3756
Block 0, Step 8, Loss: 34.7880
Block 0, Step 9, Loss: 34.2314


/Users/debangandey/Documents/GitHub/SpatialReg-Unlinked/src/Experiments/Diff_piX_piS/VIGP_Unlinked_diffpiX_diffpiS.py:225: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  M_X_star[block_idx] = torch.tensor(model_piX[block_idx].current_M_X_star).clone().data
/Users/debangandey/Documents/GitHub/SpatialReg-Unlinked/src/Experiments/Diff_piX_piS/VIGP_Unlinked_diffpiX_diffpiS.py:226: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  V_X_star[block_idx] = torch.tensor(model_piX[block_idx].current_V_X_star).clone().data


Block 1, Step 0, Loss: 37.9173
Block 1, Step 1, Loss: 37.2928
Block 1, Step 2, Loss: 36.6860
Block 1, Step 3, Loss: 36.0833
Block 1, Step 4, Loss: 35.4832
Block 1, Step 5, Loss: 34.9047
Block 1, Step 6, Loss: 34.3373
Block 1, Step 7, Loss: 33.7845
Block 1, Step 8, Loss: 33.2455
Block 1, Step 9, Loss: 32.7265
Block 2, Step 0, Loss: 41.5154
Block 2, Step 1, Loss: 40.8152
Block 2, Step 2, Loss: 40.1037
Block 2, Step 3, Loss: 39.3885
Block 2, Step 4, Loss: 38.7783
Block 2, Step 5, Loss: 38.1503
Block 2, Step 6, Loss: 37.5525
Block 2, Step 7, Loss: 36.9498
Block 2, Step 8, Loss: 36.3818
Block 2, Step 9, Loss: 35.8339
Block 3, Step 0, Loss: 41.3601
Block 3, Step 1, Loss: 40.7397
Block 3, Step 2, Loss: 40.1161
Block 3, Step 3, Loss: 39.4889
Block 3, Step 4, Loss: 38.8786
Block 3, Step 5, Loss: 38.2794
Block 3, Step 6, Loss: 37.6860
Block 3, Step 7, Loss: 37.1033
Block 3, Step 8, Loss: 36.5538
Block 3, Step 9, Loss: 36.0139
Block 4, Step 0, Loss: 39.9029
Block 4, Step 1, Loss: 39.2360
Block 4,

/Users/debangandey/Documents/GitHub/SpatialReg-Unlinked/src/Experiments/Diff_piX_piS/VIGP_Unlinked_diffpiX_diffpiS.py:250: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  M_S_star[block_idx] = torch.tensor(model_piS[block_idx].current_M_S_star).clone().data
/Users/debangandey/Documents/GitHub/SpatialReg-Unlinked/src/Experiments/Diff_piX_piS/VIGP_Unlinked_diffpiX_diffpiS.py:251: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  V_S_star[block_idx] = torch.tensor(model_piS[block_idx].current_V_S_star).clone().data
  2%|▏         | 1/50 [03:21<2:44:31, 201.46s/it]

...

 36%|███▌      | 18/50 [1:03:43<1:51:24, 208.89s/it]

Block 48, Step 9, Loss: 27.3733
Iter 18/50 | mu_lambda_beta: 3.0607 | 
 sigmasq_lambda_beta: 0.0915 | 
 lambda_a1: 98.1000 | lambda_b1: 447.8188 | lambda_a2: 98.1000 | lambda_b2: 671.9417
‣  E[ϕ]: 1.9133 | ‣ ||mu_W||: 34.0726
Average correct permutations recognized for piX across all blocks: 2.0408163265306123
Average correct permutations recognized for piS across all blocks: 1.1020408163265305
Total Loss: 2.6201
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 1.9214e-01
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.1921e-07
Block 0, Step 0, Loss: 26.1615
Block 0, Step 1, Loss: 26.1605
Block 0, Step 2, Loss: 26.1595
Block 0, Step 3, Loss: 26.1585
Block 0, Step 4, Loss: 26.1576
Block 0, Step 5, Loss: 26.1566
Block 0, Step 6, Loss: 26.1557
Block 0, Step 7, Loss: 26.1547
Block 0, Step 8, Loss: 26.1538
Block 0, Step 9, Loss: 26.1529
Block 1, Step 0, Loss: 24.6252
Block 1, Step 1, Loss: 24.6242
Block 1, Step 2, Loss: 24.6232
Block 1, Step 3, Loss: 24.6222
Block 1, Step 4

 36%|███▌      | 18/50 [1:04:31<1:54:43, 215.10s/it]

Block 21, Step 9, Loss: 27.4216


KeyboardInterrupt: 

In [ ]:
model = GPModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(3000)):
    optimizer.zero_grad()
    loss = model(s, x, y)
    loss.backward()
    optimizer.step()
    
    # Constrain sigmasq, length_scale, and tausq to be positive
    with torch.no_grad():
        model.sigmasq.clamp_(min=1e-6)
        model.phi.clamp_(min=1e-6)
        model.tausq.clamp_(min=1e-6)
        

In [ ]:
model.phi.item(), model.sigmasq.item(), model.tausq.item(), model.nu.item(), model.beta.item()

In [ ]:
model = GPArealModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(3000)):
    optimizer.zero_grad()
    loss = model(s_jumbled_within_regions, region_assignments, xbar, ybar)
    loss.backward()
    optimizer.step()
